<!-- TARGET: public course-materials repo, sessions/m08/demo.ipynb (follow-along: the Your turn cells are blank). Copy to work/m08/ before editing. -->
# Module 8 · The course project, end to end

**Python for Data Science · ESADE MIBA · Wed 25 Nov 2026, 14:00**

> **Do late deliveries hurt review scores, and where should operations act first?**

- **Stakeholder:** the operations director.
- **Decision:** where to spend next quarter's logistics effort first.
- **Evidence:** the review score of late vs on-time orders; where late orders concentrate (by
  state, by route, by seller).

You have built this case piece by piece since M04: clean tables (M04), the order mart (M05), the
charts (M06–M07). Today we assemble it, add the sellers, and end with a recommendation in one
paragraph. Nothing here needs this morning's lab.

Every number in this notebook comes from `data/raw/` through the mart, so **Restart and Run
All** reproduces it from a fresh clone. That is the point.

## 0. Setup

The mart from M05 (`data/mart/orders_enriched.parquet`), the map from M07, the ESADE chart style.
No mart? Copy `sessions/m05/lab-solution.ipynb` to `work/m05/`, Run All, and come back.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

RAW = Path("../../data/raw")
MART_FILE = Path("../../data/mart/orders_enriched.parquet")
GEO_FILE = Path("../../data/geo/brazil-states.geojson")

assert MART_FILE.exists(), "No order mart. Run sessions/m05/lab-solution.ipynb from work/m05/ first."
assert GEO_FILE.exists(), "No map file. Pull the course repository."
mart = pd.read_parquet(MART_FILE)
with open(GEO_FILE) as geo_file:
    brazil_states = json.load(geo_file)

ESADE_BLUE = "#00205B"
GREY = "#B8BEC8"
pio.templates["esade"] = go.layout.Template(
    layout=dict(
        font=dict(family="Arial", color="#2B2B2B"),
        title=dict(font=dict(color=ESADE_BLUE, size=20)),
        colorway=[ESADE_BLUE, "#4A9EFF", "#E07A1F", "#7A869A", "#2CA58D", "#C1272D"],
    )
)
pio.templates.default = "plotly_white+esade"

## 1. Review the mart, then add the sellers

### What one row is

Before any analysis: what is one row, and is it true? Raw → clean → mart: `data/raw/` is never
touched; `data/clean/` holds one typed table per entity; the mart is one row per **order**, ready
for this question.

In [ ]:
assert mart["order_id"].is_unique, "one row per order"
assert len(mart) == 99441, "every order, none lost on the way"
mart.dtypes

`is_late` is empty for orders that were never delivered: lateness only exists for a delivery.
The analysis uses delivered orders with a delivery date.

In [ ]:
delivered = mart[(mart["order_status"] == "delivered") & mart["is_late"].notna()]
print(len(delivered), "delivered orders with a delivery date")
print(f"{delivered['is_late'].mean():.1%} delivered late")

### Add the sellers

Operations works with **sellers**, not customers, so the question needs the seller of each order.
Sellers live on the **item lines**. One order, several items, possibly several sellers.

*Predict:* if we join the order's sellers onto the delivered orders with
`validate="one_to_one"`, what happens?

In [ ]:
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

order_sellers = (
    items[["order_id", "seller_id"]]
    .drop_duplicates()
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one")
)
try:
    delivered.merge(order_sellers, on="order_id", how="left", validate="one_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

`validate` stopped a silent fan-out: some orders have more than one seller. How many?

*Your turn (3 min):* count the sellers of each order in `order_sellers` and build
`single_seller_ids`: the `order_id`s with **exactly one** seller. Print how many orders have more
than one.

In [ ]:
sellers_per_order = ...
single_seller_ids = ...

In [ ]:
sellers_per_order = order_sellers.groupby("order_id")["seller_id"].nunique()
single_seller_ids = sellers_per_order[sellers_per_order == 1].index
multi_seller = (sellers_per_order > 1).sum()
print(f"{multi_seller} of {len(sellers_per_order)} orders with items have several sellers ({multi_seller / len(sellers_per_order):.1%})")

1.3% of orders. Two honest options: an analysis table at the grain **order × seller** (an order
with two sellers counts for both), or keep the **single-seller orders** and say so. We take the
second: it keeps one row per order, and we check what it costs.

In [ ]:
single_seller_orders = order_sellers[order_sellers["order_id"].isin(single_seller_ids)]
analysis = delivered[delivered["order_id"].isin(single_seller_ids)].merge(
    single_seller_orders, on="order_id", how="left", validate="one_to_one"
)
assert analysis["order_id"].is_unique
assert analysis["seller_state"].notna().all()

kept_late = analysis["is_late"].sum()
all_late = delivered["is_late"].sum()
print(f"{len(analysis)} of {len(delivered)} delivered orders kept; {kept_late} of {all_late} late orders kept")

We keep 98.7% of the orders and 99.8% of the late ones. The rule is written down, its cost is
measured, and the `assert`s prove the grain. That is what a defensible choice looks like.

## 2. Analysis → charts → recommendation

### Does lateness hurt reviews?

In [ ]:
reviewed = delivered[delivered["review_score"].notna()]
score_gap = reviewed.groupby("is_late").agg(
    orders=("order_id", "size"),
    mean_score=("review_score", "mean"),
    one_star_share=("review_score", lambda scores: (scores == 1).mean()),
)
score_gap.round(3)

Yes, and by a lot: 2.27 vs 4.29 on average, and **54% of late orders get one star** (7% of
on-time ones). That's the "why": now the "where".

### Where: by customer state (M07)

In [ ]:
by_customer_state = (
    delivered.groupby("customer_state")
    .agg(orders=("order_id", "size"), late_orders=("is_late", "sum"), late_share=("is_late", "mean"))
    .reset_index()
)
fig_map = px.choropleth(
    by_customer_state,
    geojson=brazil_states,
    locations="customer_state",
    featureidkey="properties.sigla",
    color="late_share",
    color_continuous_scale="Reds",
    hover_data=["orders", "late_orders"],
    title="The North-East has the highest share of late deliveries",
    labels={"late_share": "Share late"},
)
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map

High **share**, low **volume**: Alagoas has 397 delivered orders. Fixing Alagoas fixes 85 late
orders. Operations needs both numbers: where the risk is, and where the late orders are.

### Where: by route (seller state → customer state)

A delivery goes **from** a seller **to** a customer. A route is the pair.

*Your turn (4 min):* add a `route` column to `analysis` (`"SP → RJ"`: seller state, an arrow,
customer state). Then build `routes`: one row per route with `orders`, `late_orders` and
`late_share`.

In [ ]:
analysis["route"] = ...
routes = ...

In [ ]:
analysis["route"] = analysis["seller_state"] + " → " + analysis["customer_state"]
routes = (
    analysis.groupby("route")
    .agg(orders=("order_id", "size"), late_orders=("is_late", "sum"), late_share=("is_late", "mean"))
    .reset_index()
)
assert routes["orders"].sum() == len(analysis), "every order on exactly one route"
print(len(routes), "routes")
routes.sort_values("late_orders", ascending=False).head(8)

São Paulo → São Paulo has the most late orders, but only because it has 30,000 orders: 4.7% late.
São Paulo → Rio de Janeiro: 8,048 orders, **14.3% late**, three times the rate.

One chart for both questions: volume on x, risk on y, the size of the problem as the bubble.
Only routes with at least 500 orders, so that every share rests on enough orders.

In [ ]:
big_routes = routes[routes["orders"] >= 500]
fig_routes = px.scatter(
    big_routes,
    x="orders",
    y="late_share",
    size="late_orders",
    hover_name="route",
    log_x=True,
    title="São Paulo → Rio de Janeiro: high volume and high risk",
    labels={"orders": "Delivered orders on the route (log scale)", "late_share": "Share delivered late"},
)
fig_routes.update_layout(yaxis_tickformat=".0%")
fig_routes

Top right is where volume meets risk: **SP → RJ**. Its 1,147 late orders are 18% of all late
orders. Bottom right (SP → SP) is big but healthy. `log_x` spreads the routes: say so on the
axis.

**Check the chart** before anyone acts on it:

In [ ]:
plotted = pd.Series(list(fig_routes.data[0].y), index=list(fig_routes.data[0].hovertext))
expected = analysis.groupby("route")["is_late"].mean()
assert ((plotted - expected[plotted.index]).abs() < 1e-9).all()
assert len(plotted) == (routes["orders"] >= 500).sum()
print(f"SP → RJ: {routes.set_index('route').loc['SP → RJ', 'late_orders'] / all_late:.0%} of all late orders")

### Where: by seller

A route is many sellers. Do a few sellers cause most of the late orders?

In [ ]:
by_seller = analysis.groupby("seller_id").agg(orders=("order_id", "size"), late_orders=("is_late", "sum"))
top_sellers_late = by_seller["late_orders"].nlargest(100).sum()
print(f"{len(by_seller)} sellers; the 100 with the most late orders account for {top_sellers_late / kept_late:.0%} of them")

Half of the late orders come from 100 of about 2,950 sellers. That's a list operations can call.

### The recommendation

One paragraph, for the operations director: **finding, size, action, caveat.**

> Late deliveries cost about two review stars (2.3 vs 4.3), and more than half of late orders
> get one star. Act first on **São Paulo sellers shipping to Rio de Janeiro**: one route, 14% late
> (three times São Paulo's local rate), 18% of all late orders. Second, start a programme with
> the **100 sellers** behind half of the late orders. The North-East has the highest late share
> but few orders: monitor it, don't start there. *Caveat:* 1.3% of orders with several sellers are
> left out of the route and seller analysis; it keeps 99.8% of the late orders.

*Your turn (5 min):* write your own version in the cell below. Use your numbers, not mine. One
paragraph. Would the director know what to do on Monday?

*Your recommendation:* ...

### Reproducible?

Restart the kernel and Run All. Every number above comes back, from `data/raw/` through the
mart, with no number typed by hand. That is what the exam's first question asks about, and what
the hackathon marks.